# 01 Finding and downloading Biomass ocean data

Biomass only recently started acquiring over the ocean, and ocean scenes are a best effort secondary objective, so the first job was finding out what's actually there. This notebook does three things:

1. **Search and download.** Log in to ESA MAAP, search the `BiomassLevel1a` STAC collection by area and date (or paste granule IDs straight from the MAAP explorer), and download only the files that aren't already on disk.
2. **Scan the whole catalogue.** Go through every Level-1a granule, keep one record per unique swath location, and classify each as ocean, coastal or land from how much of its footprint is sea. This is how the ocean scenes were found in the first place.
3. **Reference data for a scene.** SMOS, the CMEMS salinity and wave analyses, OSTIA SST, ERA5 wind, IMERG rain and SMAP, cut to the scene and the day.

Before running: a MAAP account (explorer.maap.eo.esa.int) and an offline token, a Copernicus Marine account (`copernicusmarine login` once), a CDS account with `~/.cdsapirc` for ERA5, and a NASA Earthdata login for IMERG and SMAP. The MAAP credentials go in a small text file that is **never** committed (it's in `.gitignore`):

```
maap_offline_token = ...
maap_client_id = ...
maap_client_secret = ...
```

In [ ]:
# ---- the only cell that needs editing ----
from pathlib import Path

CREDS = Path(r"C:\Users\Orlaith.Doyle\eo_credentials.txt")
DATA = Path(r"C:\Users\Orlaith.Doyle")                       # products end up straight in here
REFS = Path(r"C:\Users\Orlaith.Doyle\refs")                  # reference data for the scenes
SCAN_DIR = Path(r"C:\Users\Orlaith.Doyle\swath_scan")

CATALOGUE = "https://catalog.maap.eo.esa.int/catalogue/"
COLLECTION = "BiomassLevel1a"
TOKEN_URL = "https://iam.maap.eo.esa.int/realms/esa-maap/protocol/openid-connect/token"

# search: either paste granule IDs from the MAAP explorer, or leave empty and search by box + dates
TARGET_GRANULE_IDS = []
BBOX = [-52.0, -1.0, -46.0, 6.0]            # lon min, lat min, lon max, lat max (Amazon mouth)
DATES = "2026-07-20/2026-07-31"
PRODUCT_TYPE = "SCS__1S"                     # single look complex, slant range

# catalogue scan
SCAN_START, SCAN_END = "2025-11-01", "2026-10-31"
BATCH_SIZE = 2000                            # checkpoint every this many granules
OCEAN_MIN, LAND_MAX = 0.9, 0.1               # ocean fraction >= 0.9 ocean, <= 0.1 land, coastal in between

In [ ]:
import sys, json, time, zipfile
sys.path.append(str(Path.cwd()))
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from pystac_client import Client
import bm_tools as bm

for p in (DATA, REFS, SCAN_DIR):
    p.mkdir(parents=True, exist_ok=True)

creds = bm.load_creds(CREDS)
TOKEN = bm.get_token(creds, TOKEN_URL)
HEAD = {"Authorization": f"Bearer {TOKEN}"}
print("logged in to MAAP")


def open_cat():
    return Client.open(CATALOGUE, headers=HEAD)


def search(cat, **kw):
    """the catalogue throws the odd 502, so retry a few times before giving up.
    method GET and the collection name are both needed or it quietly returns nothing"""
    for attempt in range(5):
        try:
            return list(cat.search(collections=[COLLECTION], method="GET", **kw).items())
        except Exception as e:
            print(f"  search failed ({type(e).__name__}), retry {attempt + 1}/5")
            time.sleep(5 * (attempt + 1))
    raise RuntimeError("catalogue still failing after 5 tries")

## 1. Search

If granule IDs were pasted in they're used as they are. Otherwise the catalogue is searched by box and dates, day by day (a long date range in one go was where the 502 errors mostly happened), and only single look complex products are kept, since everything later needs the phase.

In [ ]:
cat = open_cat()
if TARGET_GRANULE_IDS:
    items = search(cat, ids=TARGET_GRANULE_IDS)
else:
    d0, d1 = DATES.split("/")
    items = []
    days = pd.date_range(d0, d1, freq="D")
    for i, day in enumerate(days):
        items += search(cat, bbox=BBOX, datetime=f"{day:%Y-%m-%d}T00:00:00Z/{day:%Y-%m-%d}T23:59:59Z")
        print(f"  day {i + 1}/{len(days)} {day:%Y-%m-%d}: {len(items)} so far")
items = [it for it in items if PRODUCT_TYPE in it.id]
TARGET_GRANULE_IDS = sorted({it.id for it in items})

for it in items:
    m = bm.parse_name(it.id)
    print(f"{it.id[:60]}  T{m['track']:03d} F{m['frame']:03d}  {it.properties.get('sat:orbit_state', '')}")
print(len(items), "granules")

## 2. Download what's missing

Each granule goes in its own folder named after it. Before downloading, the folder is checked and any file that's already there with the right size is skipped, so a half finished download can just be run again. If MAAP hands back a zip it's unpacked in place.

This replaced downloading full zips by hand from the MAAP explorer, which was fine for a handful of granules but not for a few hundred.

In [ ]:
def fetch(url, dest, chunk=1 << 20):
    tmp = dest.with_suffix(dest.suffix + ".part")
    with requests.get(url, headers=HEAD, stream=True, timeout=120) as r:
        if r.status_code == 401:
            raise RuntimeError("401, the token has expired. rerun the login cell")
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        done, last = 0, 0
        with open(tmp, "wb") as f:
            for b in r.iter_content(chunk):
                f.write(b); done += len(b)
                if total and done - last > total / 10:
                    print(f"      {100 * done / total:3.0f} %", end="\r"); last = done
    tmp.rename(dest)
    return total


for n, it in enumerate(items):
    folder = DATA / it.id
    folder.mkdir(exist_ok=True)
    print(f"[{n + 1}/{len(items)}] {it.id}")
    for key, a in it.assets.items():
        name = Path(a.href.split("?")[0]).name
        dest = folder / name
        size = a.extra_fields.get("file:size")
        if dest.exists() and (size is None or dest.stat().st_size == size):
            continue
        for attempt in range(3):
            try:
                fetch(a.href, dest)
                break
            except requests.RequestException as e:
                print(f"    {name}: {e}, retrying")
                time.sleep(10)
        if dest.suffix == ".zip" and dest.exists():
            with zipfile.ZipFile(dest) as z:
                z.extractall(folder)
    print("    done")

## 3. Scanning the whole catalogue

To find ocean scenes at all, every granule in the collection is listed and reduced to one record per unique swath location (the `grid:code` property). Repeat visits of the same location just add to that record's count and date list.

It's slow, hours for the full archive, so it's built to be stopped and restarted. It works one month at a time, writes a checkpoint every `BATCH_SIZE` granules, and remembers which months are finished in `scan_state.json`. Any API error is treated as a dropped connection (reconnect and redo that month), never as the end of the collection. Early on a dropped connection got read as "no more results" and the scan stopped short without saying so.

In [ ]:
state_f = SCAN_DIR / "scan_state.json"
swath_f = SCAN_DIR / "biomass_all_swaths.csv"
state = json.loads(state_f.read_text()) if state_f.exists() else {"done_months": [], "n_granules": 0}
sw = pd.read_csv(swath_f).set_index("grid_code").to_dict("index") if swath_f.exists() else {}


def checkpoint():
    pd.DataFrame.from_dict(sw, orient="index").rename_axis("grid_code").to_csv(swath_f)
    state_f.write_text(json.dumps(state))


months = pd.date_range(SCAN_START, SCAN_END, freq="MS")
t_start = time.time()
for mo in months:
    key = f"{mo:%Y-%m}"
    if key in state["done_months"]:
        continue
    end = (mo + pd.offsets.MonthEnd(1)).strftime("%Y-%m-%dT23:59:59Z")
    while True:
        n_mo = 0
        try:
            cat = open_cat()
            for it in cat.search(collections=[COLLECTION], method="GET",
                                 datetime=f"{mo:%Y-%m-%d}T00:00:00Z/{end}").items():
                code = it.properties.get("grid:code") or it.id
                day = (it.properties.get("datetime") or "")[:10]
                if code not in sw:
                    lon0, lat0, lon1, lat1 = it.bbox
                    sw[code] = {"first_id": it.id, "geometry": json.dumps(it.geometry), "n_visits": 0, "dates": "",
                                "lon": (lon0 + lon1) / 2, "lat": (lat0 + lat1) / 2}
                sw[code]["n_visits"] += 1
                if day not in sw[code]["dates"]:
                    sw[code]["dates"] += (" " if sw[code]["dates"] else "") + day
                n_mo += 1
                state["n_granules"] += 1
                if state["n_granules"] % BATCH_SIZE == 0:
                    checkpoint()
                    print(f"  {state['n_granules']} granules, {len(sw)} unique swaths, "
                          f"{(time.time() - t_start) / 3600:.1f} h")
            break
        except Exception as e:
            # never read an error as the end of the collection, reconnect and redo the month
            print(f"  {key}: {type(e).__name__} after {n_mo} granules, reconnecting in 30 s")
            time.sleep(30)
            HEAD = {"Authorization": f"Bearer {bm.get_token(creds, TOKEN_URL)}"}
    state["done_months"].append(key)
    checkpoint()
    print(f"{key} done: {n_mo} granules, {len(sw)} unique swaths so far")

print(f"scan finished: {state['n_granules']} granules, {len(sw)} unique swath locations")

## 4. Ocean, coastal or land

Each unique footprint is intersected with the Natural Earth land polygons (1:50 m) and the ocean fraction is 1 minus the land share of its area. At least `OCEAN_MIN` is ocean, at most `LAND_MAX` is land, and everything in between is coastal.

Footprints that cross the antimeridian get their western longitudes shifted by 360 so the polygon doesn't wrap the whole planet, and the land polygons are checked shifted too.

In [ ]:
import cartopy.io.shapereader as shpreader
import cartopy.crs as ccrs
from shapely.geometry import shape
from shapely.affinity import translate
from shapely.ops import unary_union
from shapely.strtree import STRtree

land_geoms = list(shpreader.Reader(shpreader.natural_earth("50m", "physical", "land")).geometries())
land_geoms += [translate(g, 360) for g in land_geoms]       # copies for footprints shifted past 180
tree = STRtree(land_geoms)


def ocean_fraction(geojson):
    g = shape(json.loads(geojson))
    xs = np.array(g.exterior.coords)[:, 0] if g.geom_type == "Polygon" else np.array([])
    if xs.size and xs.max() - xs.min() > 180:                 # antimeridian
        c = np.array(g.exterior.coords)
        c[c[:, 0] < 0, 0] += 360
        g = shape({"type": "Polygon", "coordinates": [c.tolist()]})
    hits = tree.query(g)
    if len(hits) == 0:
        return 1.0
    land = unary_union([land_geoms[i] for i in hits]).intersection(g)
    return 1 - land.area / g.area


df = pd.DataFrame.from_dict(sw, orient="index").rename_axis("grid_code").reset_index()
fr = []
for i, gj in enumerate(df.geometry):
    fr.append(ocean_fraction(gj))
    if i % 500 == 0:
        print(f"  classified {i}/{len(df)}")
df["ocean_fraction"] = fr
df["class"] = np.where(df.ocean_fraction >= OCEAN_MIN, "ocean", np.where(df.ocean_fraction <= LAND_MAX, "land", "coastal"))
df.drop(columns="geometry").to_csv(SCAN_DIR / "biomass_swaths_classified.csv", index=False)
print(df["class"].value_counts())

fig = plt.figure(figsize=(14, 7))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.coastlines(lw=0.4); ax.set_global()
for c, col in [("land", "tab:green"), ("coastal", "tab:orange"), ("ocean", "tab:blue")]:
    s = df[df["class"] == c]
    ax.scatter(s.lon, s.lat, s=3, color=col, transform=ccrs.PlateCarree(), label=f"{c} ({len(s)})")
ax.legend(loc="lower left", markerscale=4)
ax.set_title(f"Biomass Level-1a: {len(df)} unique swath locations from {state['n_granules']} granules")
fig.savefig(SCAN_DIR / "swaths_world_classified.png", dpi=200, bbox_inches="tight")
plt.show()

## 5. Reference data for a scene

Everything Biomass gets compared against, cut to a box around the scene (with a margin so interpolation has room) and the day of the pass. Each block is independent, so run only the ones that are needed.

- **SMOS L3** daily ascending and descending maps (0.25 deg) with a per pixel acquisition time, the main salinity reference
- **CMEMS global analysis** daily salinity and temperature at 1/12 deg, a model constrained by observations
- **OSTIA** L4 SST, daily, 0.05 deg
- **CMEMS MFWAM** waves every 3 h at 1/12 deg, split into wind sea and two swells
- **ERA5** hourly 10 m wind from the Climate Data Store
- **IMERG v07** half hourly rain, Final run if it exists, otherwise Late, otherwise Early
- **SMAP** RSS v6 8 day running mean

Product and dataset IDs change now and then. If one fails, check the ID in the Copernicus Marine or Earthdata catalogue first.

In [ ]:
SCENE_BOX = [-51.0, 0.0, -45.0, 6.0]       # lon min, lat min, lon max, lat max, with a margin
SCENE_DAY = "2026-07-26"
TAG = "amazon_20260726"

import copernicusmarine as cm

def cmems(dataset_id, variables, name, hours=None):
    t0 = pd.Timestamp(SCENE_DAY) - pd.Timedelta(hours=hours or 0)
    t1 = pd.Timestamp(SCENE_DAY) + pd.Timedelta(hours=24 + (hours or 0))
    cm.subset(dataset_id=dataset_id, variables=variables,
              minimum_longitude=SCENE_BOX[0], maximum_longitude=SCENE_BOX[2],
              minimum_latitude=SCENE_BOX[1], maximum_latitude=SCENE_BOX[3],
              start_datetime=t0.isoformat(), end_datetime=t1.isoformat(),
              output_directory=str(REFS), output_filename=f"{name}_{TAG}.nc")
    print("got", name)

cmems("cmems_obs-mob_glo_phy-sss_mynrt_smos-asc_P1D", None, "smos_asc")
cmems("cmems_obs-mob_glo_phy-sss_mynrt_smos-des_P1D", None, "smos_des")
cmems("cmems_mod_glo_phy-so_anfc_0.083deg_P1D-m", ["so"], "cmems_so")
cmems("cmems_mod_glo_phy-thetao_anfc_0.083deg_P1D-m", ["thetao"], "cmems_thetao")
cmems("METOFFICE-GLO-SST-L4-NRT-OBS-SST-V2", ["analysed_sst"], "ostia")
cmems("cmems_mod_glo_wav_anfc_0.083deg_PT3H-i",
      ["VHM0", "VTPK", "VMDR", "VHM0_WW", "VTM01_WW", "VMDR_WW",
       "VHM0_SW1", "VTM01_SW1", "VMDR_SW1", "VHM0_SW2", "VTM01_SW2", "VMDR_SW2"], "waves", hours=3)

In [ ]:
# ERA5, needs ~/.cdsapirc
import cdsapi

day = pd.Timestamp(SCENE_DAY)
cdsapi.Client().retrieve("reanalysis-era5-single-levels", {
    "product_type": ["reanalysis"],
    "variable": ["10m_u_component_of_wind", "10m_v_component_of_wind"],
    "year": [f"{day:%Y}"], "month": [f"{day:%m}"], "day": [f"{day:%d}"],
    "time": [f"{h:02d}:00" for h in range(24)],
    "area": [SCENE_BOX[3], SCENE_BOX[0], SCENE_BOX[1], SCENE_BOX[2]],     # N, W, S, E
    "data_format": "netcdf"}, str(REFS / f"era5_wind_{TAG}.nc"))
print("got ERA5")

In [ ]:
# IMERG half hourly and SMAP 8 day, NASA Earthdata
import earthaccess

earthaccess.login(persist=True)
imerg_dir = REFS / "imerg"; imerg_dir.mkdir(exist_ok=True)
day = pd.Timestamp(SCENE_DAY)
for run in ("GPM_3IMERGHH", "GPM_3IMERGHHL", "GPM_3IMERGHHE"):          # Final, Late, Early
    res = earthaccess.search_data(short_name=run, version="07",
                                  temporal=(f"{day:%Y-%m-%d}T00:00:00", f"{day:%Y-%m-%d}T23:59:59"))
    if res:
        earthaccess.download(res, str(imerg_dir))
        print(f"IMERG {run}: {len(res)} files")
        break

smap_dir = REFS / "smap"; smap_dir.mkdir(exist_ok=True)
res = earthaccess.search_data(short_name="SMAP_RSS_L3_SSS_SMI_8DAY-RUNNINGMEAN_V6",
                              temporal=(f"{day:%Y-%m-%d}", f"{day:%Y-%m-%d}"))
earthaccess.download(res, str(smap_dir))
print("SMAP:", len(res), "files")